# StreaMAX Fit — GPU Nested Sampling (dynesty)
Run on Colab with T4 GPU runtime.

In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 2. Set paths — edit these to match your Drive layout
REPO_DIR = '/content/drive/MyDrive/Vasily/StreaMAX_fit'  # path to the repo on Drive
CSV_PATH = '/content/drive/MyDrive/Vasily/StreaMAX_fit/Au26_stream_4664.csv'  # path to data

import os
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')
print(f'Files: {os.listdir(".")}')

Working directory: /content/drive/MyDrive/Vasily/StreaMAX_fit
Files: ['play.ipynb', 'prior.py', 'llikelihood.py', 'main.py', 'fit.py', 'Au26_stream_4664.csv', 'uv.lock', 'run_colab.ipynb', 'pyproject.toml', '__pycache__', 'utils.py']


In [3]:
# 3. Install dependencies
!pip install -q streamax dynesty corner matplotlib pandas numpy scipy tqdm

In [4]:
# 4. Check GPU
import jax
print(f'JAX devices: {jax.devices()}')
assert any('gpu' in str(d).lower() or 'cuda' in str(d).lower() for d in jax.devices()), \
    'No GPU found! Change runtime to GPU.'

JAX devices: [CudaDevice(id=0)]


In [5]:
# 5. Settings — edit as needed
TRIAXIAL = False        # True for triaxial, False for axisymmetric
N_PARTICLES = 10000
N_MIN = 3
VAR_RATIO = 1e-10
N_BINS = 18
THETA_MIN = -1
THETA_MAX = 1
MIN_COUNT = 3
N_LIVE = 2000
OUTPUT_DIR = 'results'

In [6]:
import numpy as np
import jax.numpy as jnp
from main import load_stream_data
dict_data = load_stream_data(CSV_PATH, theta_min=THETA_MIN*np.pi, theta_max=THETA_MAX*np.pi, n_bins=N_BINS, min_count=MIN_COUNT)

In [7]:
from fit import dynesty_fit
from llikelihood import *
from prior import *

if TRIAXIAL:
    ndim = 16
    prior_fn = prior_transform_tri
    mode = 'triaxial'
else:
    ndim = 14
    prior_fn = prior_transform_axi
    mode = 'axisymmetric'

p = np.random.uniform(0, 1, size=ndim)
params = prior_transform_axi(p)

def logl(params, dict_data, n_particles=20000, n_min=3, var_ratio_thresh=9.0, triaxial=False, unroll=True):
    theta_stream, r_stream, xv_stream = params_to_stream(params, n_particles, triaxial=triaxial, unroll=unroll)
    r_bin, sig_bin, count_bin = jax.vmap(StreaMAX.get_track_2D, in_axes=(None, None, 0, None))(
        theta_stream, r_stream, dict_data['theta'], dict_data['bin_width'])

    sig_idx = 15 if triaxial else 13
    n_bad = jnp.sum(count_bin < n_min)
    if n_bad > 0:
        logl = BAD_VAL * n_bad

    else:
        var_data  = dict_data['r_err']**2
        var_model = sig_bin**2 / count_bin
        n_high_error = jnp.sum(var_model / var_data > 1/var_ratio_thresh)
        if n_high_error > 0:
            logl = BAD_VAL/1e3 * n_high_error

        else:
            var = var_data + params[sig_idx]**2
            logl = -.5 * jnp.sum((r_bin - dict_data['r'])**2 / var + jnp.log(2 * jnp.pi * var))

    return logl

like = logl(params, dict_data, n_particles=20000, n_min=3, var_ratio_thresh=9.0, triaxial=False, unroll=False)

In [8]:
import time
start = time.time()
like = logl(params, dict_data, n_particles=10000, n_min=3, var_ratio_thresh=9.0, triaxial=False, unroll=True)
end = time.time()
print(f'Log-likelihood: {like}, time taken: {end - start} seconds')

Log-likelihood: -1.0000000272564224e+16, time taken: 56.54475545883179 seconds


In [32]:
start = time.time()
theta_stream, r_stream, xv_stream = params_to_stream(params, N_PARTICLES, triaxial=TRIAXIAL, unroll=True)
# r_bin, sig_bin, count_bin = jax.vmap(StreaMAX.get_track_2D, in_axes=(None, None, 0, None))(theta_stream, r_stream, dict_data['theta'], dict_data['bin_width'])
end = time.time()
print(f'params_to_stream + get_track_2D: time taken: {end - start} seconds')

params_to_stream + get_track_2D: time taken: 0.009876012802124023 seconds


In [ ]:
jax.grad(logl)(params, dict_data, n_particles=10000, n_min=3, var_ratio_thresh=9.0, triaxial=False, unroll=True)

In [71]:
import dynesty
import dynesty.utils as dyut

def dynesty_fit(dict_data, logl_fn, prior_fn, ndim, n_particles=10000, n_min=101, var_ratio=9.0, nlive=2000, triaxial=False):
    logl_args = (dict_data, n_particles, n_min, var_ratio, triaxial)
    dns = dynesty.DynamicNestedSampler(logl_fn,
                            prior_fn,
                            ndim,
                            logl_args=logl_args,
                            nlive=nlive,
                            sample='rslice')
    dns.run_nested(n_effective=10000)

    res   = dns.results
    inds  = np.arange(len(res.samples))
    inds  = dyut.resample_equal(inds, weights=np.exp(res.logwt - res.logz[-1]))
    samps = res.samples[inds]
    logl  = res.logl[inds]

    dns_results = {
                    'dns': dns,
                    'samps': samps,
                    'logl': logl,
                    'logz': res.logz,
                    'logzerr': res.logzerr,
                }

    return dns_results

dict_results = dynesty_fit(dict_data, logl, prior_fn, ndim,
                            n_particles=N_PARTICLES, n_min=N_MIN,
                            var_ratio=VAR_RATIO, nlive=N_LIVE,
                            triaxial=TRIAXIAL)

0it [00:00, ?it/s]/usr/local/lib/python3.12/dist-packages/dynesty/sampler.py:199: RuntimeWarning: overflow encountered in cast
  cur_live_logl[not_finite] = _LOWL_VAL
782it [10:32,  4.45it/s, batch: 0 | bound: 0 | nc: 1 | ncall: 1061 | eff(%): 25.547 | loglstar:   -inf <   -inf <    inf | logz:   -inf +/-    nan | dlogz:    inf >  0.010]Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/dynesty/dynesty.py", line 827, in __call__
    return self.func(np.asarray(x).copy(), *self.args, **self.kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_7203/3421409978.py", line 18, in logl
    theta_stream, r_stream, xv_stream = params_to_stream(params, n_particles, triaxial=triaxial, unroll=unroll)
                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/drive/MyDrive/Vasily/StreaMAX_fit/utils.py", line 37, in params_to_stream
    _, _, xv_stream

Exception while calling loglikelihood function:
  params: [1.2817144e+01 8.4455595e+00 1.1394322e+00 6.1807193e-02 1.8486818e+00
 8.6939192e+00 1.2315660e+00 3.0498080e+02 1.9625133e+02 4.1001842e+01
 2.1931042e+01 9.5034492e+01 3.9452031e+00 2.0843906e+01]
  args: ({'theta': array([-2.96705973, -2.61799388, -2.26892803, -1.91986218, -1.22173048,
       -0.87266463, -0.52359878, -0.17453293,  0.17453293,  0.52359878,
        0.87266463,  1.22173048,  1.57079633,  1.91986218,  2.26892803,
        2.61799388,  2.96705973]), 'r': array([ 19.60033676,  17.05665251,  19.34141324,  20.98478993,
        42.1679453 ,  66.40667688,  99.60385341, 104.96662353,
        85.82470872,  54.19060634,  42.54663449,  37.88225958,
        35.25397222,  35.70006436,  34.54659492,  37.65856392,
        46.78207901]), 'r_err': array([0.5634902 , 0.66309155, 0.9652626 , 1.0041465 , 1.5454078 ,
       2.18907126, 0.846254  , 0.50139042, 0.76102179, 0.92969639,
       0.45077845, 0.53656305, 0.39748388, 2.1195

KeyboardInterrupt: 

In [ ]:
# 6. Calibration run — small run to estimate timing
#    This includes JIT compilation time (one-off cost).
#    The full run will reuse the compiled code so will be faster per sample.
import time as timer


from main import load_stream_data
from utils import params_to_stream, get_q
from fit import dynesty_fit

dict_data = load_stream_data(CSV_PATH, n_bins=N_BINS, min_count=MIN_COUNT)
print(f'Loaded {len(dict_data["theta"])} data points')

CALIB_SAMPLES = 500
print(f'\nRunning calibration with {CALIB_SAMPLES} samples (includes JIT compilation)...')
t0 = timer.time()
_ = jaxns_fit(dict_data, n_particles=N_PARTICLES, n_min=N_MIN,
              var_ratio=VAR_RATIO, max_samples=CALIB_SAMPLES,
              num_live_points=NUM_LIVE_POINTS, triaxial=TRIAXIAL)
t_calib = timer.time() - t0

# Estimate full run (compilation is ~fixed cost, sampling scales ~linearly)
ratio = MAX_SAMPLES / CALIB_SAMPLES
t_est = t_calib * ratio  # rough upper bound
print(f'\nCalibration: {t_calib:.0f}s for {CALIB_SAMPLES} samples')
print(f'Estimated full run ({MAX_SAMPLES} samples): ~{t_est/60:.0f} min (upper bound, likely faster since compilation is done)')

In [ ]:
# 7. Full run
import json
import pickle

# Create output directory
nlp_str = f'nlp{NUM_LIVE_POINTS}' if NUM_LIVE_POINTS else 'nlpAuto'
run_name = f'maxs{MAX_SAMPLES}_{nlp_str}_npart{N_PARTICLES}_nmin{N_MIN}_vr{VAR_RATIO}'
run_dir = os.path.join(OUTPUT_DIR, run_name)
os.makedirs(run_dir, exist_ok=True)

# Save hyperparameters
hyperparams = {
    'max_samples': MAX_SAMPLES, 'num_live_points': NUM_LIVE_POINTS,
    'n_particles': N_PARTICLES, 'n_min': N_MIN, 'var_ratio': VAR_RATIO,
    'n_bins': N_BINS, 'min_count': MIN_COUNT,
}
with open(os.path.join(run_dir, 'hyperparameters.json'), 'w') as f:
    json.dump(hyperparams, f, indent=2)

# Fit
mode = 'triaxial' if TRIAXIAL else 'axisymmetric'
print(f'Fitting {mode} model with jaxns on {jax.devices()[0]}')
print(f'Run directory: {run_dir}')

dict_results = jaxns_fit(
    dict_data, n_particles=N_PARTICLES, n_min=N_MIN,
    var_ratio=VAR_RATIO, max_samples=MAX_SAMPLES,
    num_live_points=NUM_LIVE_POINTS, triaxial=TRIAXIAL)

# Save
with open(os.path.join(run_dir, 'dict_results.pkl'), 'wb') as f:
    pickle.dump(dict_results, f)
with open(os.path.join(run_dir, 'dict_data.pkl'), 'wb') as f:
    pickle.dump(dict_data, f)

print(f'\nResults saved to {run_dir}')
print(f'log Z = {dict_results["log_Z_mean"]:.2f} +/- {dict_results["log_Z_uncert"]:.2f}')
print(f'ESS = {dict_results["ESS"]:.0f}')

In [ ]:
# 8. Corner plot
import corner
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 18})

LABELS_AXI = ['logM', 'Rs', 'dirx', 'diry', 'dirz',
              'logm', 'rs', 'x0', 'z0', 'vx0', 'vy0', 'vz0', 'time', 'sig']
LABELS_TRI = ['logM', 'Rs', 'p', 'q', 'dirx', 'diry', 'dirz',
              'logm', 'rs', 'x0', 'z0', 'vx0', 'vy0', 'vz0', 'time', 'sig']

labels = LABELS_TRI if TRIAXIAL else LABELS_AXI
fig_corner = corner.corner(dict_results['samps'], labels=labels, color='blue',
                           quantiles=[0.16, 0.5, 0.84], show_titles=True,
                           title_kwargs={'fontsize': 12})
fig_corner.savefig(os.path.join(run_dir, 'corner_plot.pdf'), bbox_inches='tight')
plt.show()

In [ ]:
# 9. Best fit plot
import StreaMAX

best_params = dict_results['samps'][np.argmax(dict_results['logl'])]
theta_stream, r_stream, xv_stream = params_to_stream(best_params, N_PARTICLES, triaxial=TRIAXIAL)

r_bin, _, _ = jax.vmap(StreaMAX.get_track_2D, in_axes=(None, None, 0, None))(
    theta_stream, r_stream, dict_data['theta'], dict_data['bin_width'])

dt = dict_data['delta_theta']
c, s = np.cos(dt), np.sin(dt)
x0 = np.array(xv_stream[:, 0])
y0 = np.array(xv_stream[:, 1])
x_stream = x0 * c - y0 * s
y_stream = x0 * s + y0 * c
theta_abs = dict_data['theta'] + dt
x_data = dict_data['r'] * np.cos(theta_abs)
y_data = dict_data['r'] * np.sin(theta_abs)
r_bin_np = np.array(r_bin)
x_bin = r_bin_np * np.cos(theta_abs)
y_bin = r_bin_np * np.sin(theta_abs)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].scatter(x_stream, y_stream, c='blue', s=1, alpha=0.3, label='Best fit stream')
axes[0].scatter(x_bin, y_bin, c='lime', s=40, zorder=5, label='Model track')
axes[0].scatter(x_data, y_data, c='red', s=30, zorder=6, label='Data')
axes[0].set_xlabel('X (kpc)'); axes[0].set_ylabel('Y (kpc)')
axes[0].set_aspect('equal'); axes[0].legend(fontsize=12)

axes[1].errorbar(dict_data['theta'], dict_data['r'], yerr=dict_data['r_err'],
                 fmt='o', color='red', capsize=3, label='Data')
axes[1].scatter(dict_data['theta'], r_bin_np, c='lime', s=40, zorder=5, label='Model track')
axes[1].scatter(np.array(theta_stream), np.array(r_stream), c='blue', s=1, alpha=0.1)
axes[1].set_xlabel(r'$\theta$ (rad)'); axes[1].set_ylabel('r (kpc)')
axes[1].legend(fontsize=12)

fig.tight_layout()
fig.savefig(os.path.join(run_dir, 'best_fit.pdf'), bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# 10. Flattening posterior
samps = dict_results['samps']

if TRIAXIAL:
    p_samps, q_samps = samps[:, 2], samps[:, 3]
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    axes[0].hist(p_samps, bins=30, density=True, alpha=0.7, color='blue', range=(0.5, 1.0))
    axes[0].axvline(np.median(p_samps), color='blue', ls='--', lw=2)
    axes[0].axvline(1.0, color='k', lw=2); axes[0].set_xlabel('p = b/a'); axes[0].set_ylabel('Density')
    axes[1].hist(q_samps, bins=30, density=True, alpha=0.7, color='blue', range=(0.5, 1.0))
    axes[1].axvline(np.median(q_samps), color='blue', ls='--', lw=2)
    axes[1].axvline(1.0, color='k', lw=2); axes[1].set_xlabel('q = c/a'); axes[1].set_ylabel('Density')
    axes[2].scatter(p_samps, q_samps, s=1, alpha=0.2, color='blue')
    axes[2].plot([0.5, 1.0], [0.5, 1.0], 'k--', lw=1)
    axes[2].set_xlabel('p = b/a'); axes[2].set_ylabel('q = c/a')
    axes[2].set_xlim(0.5, 1.0); axes[2].set_ylim(0.5, 1.0); axes[2].set_aspect('equal')
else:
    q_samps = get_q(samps[:, 2], samps[:, 3], samps[:, 4])
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.hist(q_samps, bins=30, density=True, alpha=0.7, color='blue', range=(0.5, 1.5))
    ax.axvline(np.median(q_samps), color='blue', ls='--', lw=2)
    ax.axvline(1.0, color='k', lw=2); ax.set_xlabel('Halo Flattening q'); ax.set_ylabel('Density')

fig.tight_layout()
fig.savefig(os.path.join(run_dir, 'flattening.pdf'), bbox_inches='tight', dpi=150)
plt.show()